# Part 6 — Your Own Selection of External Databases

*Written for idtrack 0.1.2. An advanced topic: Parts 1 to 5 use the configuration that ships with idtrack.*

The **externals configuration** of an organism is a YAML file that says which **external databases** (HGNC, MGI, UniProt, RefSeq, ...) a graph includes, per form (gene, transcript, translation), assembly and release. idtrack ships one for each of the five organisms (`idtrack/default_config/<organism>_externals_modified.yml`), made exactly as this part shows: a **template** that lists every database Ensembl has at every release and form, then `Include: true` for an allowlist. You need your own file only to:
- change the selection of databases (a file of the same name in your local repository takes the place of the shipped one);
- build a graph whose snapshot release the shipped files do not cover (they cover releases up to 116; section 6.6);
- add an organism (section 6.7).

This part covers:
- **6.1** the configuration in plain language;
- **6.2 Human** (*Homo sapiens*): two assemblies served in parallel (GRCh38 and GRCh37);
- **6.3 Mouse** (*Mus musculus*), **6.4 Pig** (*Sus scrofa*), **6.5 Rhesus and cynomolgus macaques**: one maintained assembly per release;
- **6.6** extending a configuration to a new Ensembl release (without making it again);
- **6.7** adding an organism.

What a selection does to a graph, and how Ensembl's names are filed under a selected database: [External databases](../databases.rst).

> **How this part runs.** A template is made from Ensembl's tables of every release and form of the organism (`create_database_content()`, then `create_template_yaml()`). With the tables already in the local repository this took about 2.5 hours for human, 45 minutes for mouse and 5 to 10 minutes for pig and each macaque (one process), and the first time it also downloads every table. So the walk-through below starts from the **shipped templates** (`idtrack/default_config/<organism>_externals_template.yml`), which are what those two calls write for releases up to 116, and section 6.2.2 shows the regeneration as one optional cell. Every other cell runs in seconds.

## 6.0 — Prerequisites

- idtrack installed (Part 1).
- A writable **local repository** (the folder of tables and graph files of Part 2).
- Network access only if you make a template again (6.2.2) or extend a configuration (6.6).

**What you get at the end** (one per organism you work through), in your local repository:
- `homo_sapiens_externals_modified.yml`, `mus_musculus_externals_modified.yml`, `sus_scrofa_externals_modified.yml`, `macaca_mulatta_externals_modified.yml`, `macaca_fascicularis_externals_modified.yml`.

With the allowlists as shipped, each file is byte for byte the shipped one; change an allowlist to make your own selection. A file you made can be shared with collaborators, and must be, with results built from it.

In [1]:
# Load notebook utilities (collapsible output magic for tutorials). `_notebook_utils.py` sits next to the notebooks in
# idtrack's repository; a notebook downloaded on its own runs without it and shows the logs as they come.
try:
    get_ipython().run_line_magic('load_ext', '_notebook_utils')
except ModuleNotFoundError:
    from IPython.core.magic import register_cell_magic

    @register_cell_magic
    def collapse(line, cell):
        get_ipython().run_cell(cell)

In [2]:
# Setup
from __future__ import annotations

import filecmp
import os
from pathlib import Path

import yaml

import idtrack
from idtrack import API
from idtrack.build.default_externals import DEFAULT_EXTERNALS, apply_allowlist
from idtrack.build.externals import write_externals_yaml

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)
DEFAULT_CONFIG = Path(idtrack.__file__).parent / 'default_config'  # the shipped templates and configurations

API.configure_logger()
print('Local repository:', LOCAL_REPOSITORY)


def make_configuration(organism, allowlist, template, modified):
    """Write `modified`: the template with Include: true for the allowlist, in the layout of the shipped files."""
    data = yaml.safe_load(Path(template).read_text(encoding='utf-8'))
    data, enabled, absent = apply_allowlist(data, organism, allowlist)
    write_externals_yaml(data, str(modified))
    same = filecmp.cmp(modified, DEFAULT_CONFIG / Path(modified).name, shallow=False)
    print(f'Enabled {enabled} (form, database, assembly) entries; wrote {Path(modified).name}')
    print('Allowlisted (form, database) pairs the template does not have:', absent or 'none')
    print('Same as the shipped configuration:', same)

Local repository: idtrack_cache


## 6.1 — The externals configuration in plain language

The configuration answers one question: **which external databases does the graph include?** Ensembl links its genes to many external resources; some are curated and useful, some repeat others, and some link one id to many genes. A good rule: select a curated set of strong databases, and avoid near-duplicates, which add ambiguity.

### Structure

```yaml
<organism>:
  gene:                       # the form: gene, transcript or translation
    <Database Name>:
      Assembly:
        "<assembly key>":
          Ensembl release: 90,91,92,...
          Include: false
```

**You edit one thing:** `Include: false` → `Include: true` (a YAML boolean, without quotes; idtrack refuses anything else and names the entry). Entries of an assembly the organism does not have, or of releases outside the assembly's range in idtrack's organism table, are left out with a warning.

Why the `Assembly:` level? Ensembl lists a database's links per assembly and release, and idtrack reads a database for the assemblies and releases an entry enables. For a curated database, enabling it on **every assembly the template lists** is the right default: its ids are then recognised at every release, whichever genome build a dataset was annotated on.

> **Warning:** Assembly keys are species-specific: `38` is GRCh38 for human and GRCm38 for mouse.

### The snapshot release

A graph is built up to a **snapshot release** (its last release), and the configuration must list that release. The shipped files list releases up to 116; the tutorials use 115. Fix the release in your scripts (a graph is built per snapshot release, and Ensembl publishes a new release every few months); for a project, the release of your reference annotation, or of a published dataset you integrate, are good choices.

## 6.2 — Human (*Homo sapiens*)

Human is the organism with two assemblies served in parallel: GRCh38 (`38`) from release 76 and GRCh37 (`37`) from release 55 on, and NCBI36 (`36`) up to release 54. Enabling a database on every assembly means:
- its identifiers are recognised at the releases of every assembly (GRCh37-era inputs need the `37` entries);
- a conversion to a release of an earlier assembly can return that assembly's identifiers (the final step of a conversion is made on the assembly Ensembl maintained at the target release).

The human selection holds HGNC's symbols, NCBI's gene ids, UniProt's gene names, accessions (Swiss-Prot, TrEMBL), RefSeq's accessions, CCDS, Havana, Vega and clone-based names, miRBase and Rfam, and the synonyms of the symbol and identifier databases.

In [3]:
organism_hs = idtrack.organisms.resolve_organism('human')  # offline
SNAPSHOT_RELEASE_HS = 115
organism_hs, SNAPSHOT_RELEASE_HS

('homo_sapiens', 115)

### 6.2.1 A database manager

`API.database_manager(...)` gives the object that reads Ensembl's tables into the local repository, up to the snapshot release; its `external_inst` handles the externals configuration.

In [4]:
dm_hs = API.database_manager(organism_hs, SNAPSHOT_RELEASE_HS, str(LOCAL_REPOSITORY))
template_hs = DEFAULT_CONFIG / f'{organism_hs}_externals_template.yml'  # the shipped template
modified_hs = Path(dm_hs.external_inst.file_name_modified_yaml(location='configured'))  # where your file goes
print('Template:', template_hs.name)
print('Configuration to write:', modified_hs)

2026-11-24 11:52:42 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of homo_sapiens assembly 38 (core suffix '38') for releases 76-116.


Template: homo_sapiens_externals_template.yml
Configuration to write: idtrack_cache/homo_sapiens_externals_modified.yml


### 6.2.2 Optional: make the template again from Ensembl's tables

This is how the shipped template was made. It reads Ensembl's external tables of every release and form (about 2.5 hours for human with the tables already downloaded; the first time it also downloads them), and writes `<organism>_externals_template.yml` into the local repository. Set `REGENERATE = True` to run it; the walk-through below uses the shipped template either way.

In [5]:
REGENERATE = False  # True: read every release's tables (hours for human; see above)

if REGENERATE:
    content_hs = dm_hs.create_database_content()
    dm_hs.external_inst.create_template_yaml(content_hs)
    template_hs = Path(dm_hs.external_inst.file_name_template_yaml())
print('Template used:', template_hs.name)

Template used: homo_sapiens_externals_template.yml


### 6.2.3 Select the databases

#### Option A: by hand (worth doing once)
1. Open the template (`homo_sapiens_externals_template.yml`).
2. Search for a database you care about (e.g. `HGNC Symbol`).
3. Change `Include: false` to `Include: true` for **every assembly block** it lists (restrict to some assemblies only if you want a configuration of one build on purpose).
4. Save it as `homo_sapiens_externals_modified.yml` in your local repository (idtrack looks there first).

#### Option B: an allowlist in code (reproducible)
The cell below holds the human allowlist as an editable dictionary, form → database → assembly keys. As it stands it is the shipped selection, the one the package keeps as `idtrack.build.default_externals.DEFAULT_EXTERNALS['homo_sapiens']`; edit it to make your own. NCBI's gene ids are listed on every form, since Ensembl attached them to translations up to release 62 and to genes from 63; a name that the template lacks (here NCBI's ids on transcripts) is skipped and reported. `apply_allowlist(template, organism, allowlist)` switches the entries on, and `write_externals_yaml` writes the file in the layout of the shipped ones.

In [6]:
DEFAULT_HS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [36, 37, 38],
        'Clone_based_vega_gene': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'HGNC Symbol': [36, 37, 38],
        'Havana gene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'RFAM': [36, 37, 38],
        'RFAM label': [36, 37, 38],
        'miRBase': [36, 37, 38],
        'miRBase label': [36, 37, 38],
        'UniProtKB Gene Name': [36, 37, 38],
        'Vega gene': [36, 37, 38],
        'Vega_gene': [36, 37, 38],
        'synonym_id::HGNC Symbol': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::UniProtKB Gene Name': [36, 37, 38],
    },
    'transcript': {
        'CCDS': [36, 37, 38],
        'CCDS label': [36, 37, 38],
        'Havana transcript': [36, 37, 38],
        'RefSeq_mRNA': [36, 37, 38],
        'RefSeq_mRNA_predicted': [36, 37, 38],
        'RefSeq_ncRNA': [36, 37, 38],
        'RefSeq_ncRNA_predicted': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
    },
    'translation': {
        'Havana translation': [36, 37, 38],
        'RefSeq_peptide': [36, 37, 38],
        'RefSeq_peptide_predicted': [36, 37, 38],
        'Uniprot/SPTREMBL': [36, 37, 38],
        'Uniprot/SWISSPROT': [36, 37, 38],
        'EntrezGene': [36, 37, 38],
        'NCBI gene (formerly Entrezgene)': [36, 37, 38],
        'synonym_id::NCBI gene (formerly Entrezgene)': [36, 37, 38],
    },
}

print('The shipped selection:', DEFAULT_HS_EXTERNALS == DEFAULT_EXTERNALS[organism_hs])
make_configuration(organism_hs, DEFAULT_HS_EXTERNALS, template_hs, modified_hs)

The shipped selection: True


Enabled 74 (form, database, assembly) entries; wrote homo_sapiens_externals_modified.yml
Allowlisted (form, database) pairs the template does not have: [('transcript', 'EntrezGene'), ('transcript', 'NCBI gene (formerly Entrezgene)'), ('transcript', 'synonym_id::NCBI gene (formerly Entrezgene)')]
Same as the shipped configuration: True


### 6.2.4 Check the configuration

Loading the file checks it against the organism table and the snapshot release, and says what the manager reads at its release: the selected databases of its form (gene, here) and the assemblies in play.

In [7]:
_ = dm_hs.external_inst.load_modified_yaml()
print('Selected databases at', SNAPSHOT_RELEASE_HS, ':', dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', dm_hs.external_inst.give_list_for_case('assembly'))

# NCBI36 (assembly 36) serves human releases 48 to 54 only (GRCh37 from 55): its entries are seen at an early release.
_dm_hs = dm_hs.change_release_auto_assembly(50)
_ = _dm_hs.external_inst.load_modified_yaml()
print('Selected databases at 50:', _dm_hs.external_inst.give_list_for_case('db'))
print('Assemblies in play:', _dm_hs.external_inst.give_list_for_case('assembly'))

2026-11-24 11:52:47 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of homo_sapiens assembly 37 (core suffix '37') for releases 55-116.


Selected databases at 115 : ['EntrezGene', 'HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'UniProtKB Gene Name', 'miRBase', 'miRBase label', 'synonym_id::HGNC Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)']
Assemblies in play: [37, 38]


2026-11-24 11:52:56 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of homo_sapiens assembly 36 (core suffix '36') for releases 48-54.


Selected databases at 50: ['Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'HGNC Symbol', 'Havana gene', 'synonym_id::HGNC Symbol']
Assemblies in play: [36]


## 6.3 — Mouse (*Mus musculus*)

Mouse has one maintained assembly per release: `37` = NCBIM37, `38` = GRCm38, `39` = GRCm39. Their release ranges do not overlap, so enabling a database on every listed assembly gives one assembly per release. Mouse gene symbols come from **MGI** (Mouse Genome Informatics): the selection holds `MGI Symbol` in place of HGNC's.

In [8]:
organism_mm = idtrack.organisms.resolve_organism('mouse')
SNAPSHOT_RELEASE_MM = 115
dm_mm = API.database_manager(organism_mm, SNAPSHOT_RELEASE_MM, str(LOCAL_REPOSITORY))
template_mm = DEFAULT_CONFIG / f'{organism_mm}_externals_template.yml'
modified_mm = Path(dm_mm.external_inst.file_name_modified_yaml(location='configured'))

2026-11-24 11:52:58 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of mus_musculus assembly 39 (core suffix '39') for releases 103-116.


In [9]:
DEFAULT_MM_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [37, 38, 39],
        'Clone_based_vega_gene': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'MGI Symbol': [37, 38, 39],
        'Havana gene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'RFAM': [37, 38, 39],
        'RFAM label': [37, 38, 39],
        'miRBase': [37, 38, 39],
        'miRBase label': [37, 38, 39],
        'UniProtKB Gene Name': [37, 38, 39],
        'Vega gene': [37, 38, 39],
        'Vega_gene': [37, 38, 39],
        'synonym_id::MGI Symbol': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::UniProtKB Gene Name': [37, 38, 39],
    },
    'transcript': {
        'CCDS': [37, 38, 39],
        'CCDS label': [37, 38, 39],
        'Havana transcript': [37, 38, 39],
        'RefSeq_mRNA': [37, 38, 39],
        'RefSeq_mRNA_predicted': [37, 38, 39],
        'RefSeq_ncRNA': [37, 38, 39],
        'RefSeq_ncRNA_predicted': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
    },
    'translation': {
        'Havana translation': [37, 38, 39],
        'RefSeq_peptide': [37, 38, 39],
        'RefSeq_peptide_predicted': [37, 38, 39],
        'Uniprot/SPTREMBL': [37, 38, 39],
        'Uniprot/SWISSPROT': [37, 38, 39],
        'EntrezGene': [37, 38, 39],
        'NCBI gene (formerly Entrezgene)': [37, 38, 39],
        'synonym_id::NCBI gene (formerly Entrezgene)': [37, 38, 39],
    },
}

print('The shipped selection:', DEFAULT_MM_EXTERNALS == DEFAULT_EXTERNALS[organism_mm])
make_configuration(organism_mm, DEFAULT_MM_EXTERNALS, template_mm, modified_mm)

The shipped selection: True


Enabled 80 (form, database, assembly) entries; wrote mus_musculus_externals_modified.yml
Allowlisted (form, database) pairs the template does not have: [('transcript', 'EntrezGene'), ('transcript', 'NCBI gene (formerly Entrezgene)'), ('transcript', 'synonym_id::NCBI gene (formerly Entrezgene)')]
Same as the shipped configuration: True


In [10]:
_ = dm_mm.external_inst.load_modified_yaml()
print('Selected databases:', sorted(dm_mm.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mm.external_inst.give_list_for_case('assembly')))

Selected databases: ['EntrezGene', 'MGI Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'UniProtKB Gene Name', 'miRBase', 'miRBase label', 'synonym_id::MGI Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)']
Assemblies in play: [39]


## 6.4 — Pig (*Sus scrofa*)

Pig has one maintained assembly per release: `9` = Sscrofa9, `102` = Sscrofa10.2, `111` = Sscrofa11.1. Its symbol source changes with the assembly: Sscrofa10.2 genes carry HGNC symbols, Sscrofa11.1 genes VGNC symbols, so the selection holds both. Pig datasets often mix naming conventions; a small set of strong databases matters especially here.

In [11]:
organism_ss = idtrack.organisms.resolve_organism('pig')
SNAPSHOT_RELEASE_SS = 115
dm_ss = API.database_manager(organism_ss, SNAPSHOT_RELEASE_SS, str(LOCAL_REPOSITORY))
template_ss = DEFAULT_CONFIG / f'{organism_ss}_externals_template.yml'
modified_ss = Path(dm_ss.external_inst.file_name_modified_yaml(location='configured'))

2026-11-24 11:53:00 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of sus_scrofa assembly 111 (core suffix '111') for releases 90-116.


In [12]:
DEFAULT_SS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [9, 102, 111],
        'Clone_based_vega_gene': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'HGNC Symbol': [9, 102, 111],
        'VGNC Symbol': [9, 102, 111],
        'Havana gene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'RFAM': [9, 102, 111],
        'RFAM label': [9, 102, 111],
        'miRBase': [9, 102, 111],
        'miRBase label': [9, 102, 111],
        'UniProtKB Gene Name': [9, 102, 111],
        'synonym_id::HGNC Symbol': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::UniProtKB Gene Name': [9, 102, 111],
    },
    'transcript': {
        'Havana transcript': [9, 102, 111],
        'RefSeq_mRNA': [9, 102, 111],
        'RefSeq_mRNA_predicted': [9, 102, 111],
        'RefSeq_ncRNA': [9, 102, 111],
        'RefSeq_ncRNA_predicted': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
    },
    'translation': {
        'RefSeq_peptide': [9, 102, 111],
        'RefSeq_peptide_predicted': [9, 102, 111],
        'Uniprot/SPTREMBL': [9, 102, 111],
        'Uniprot/SWISSPROT': [9, 102, 111],
        'EntrezGene': [9, 102, 111],
        'NCBI gene (formerly Entrezgene)': [9, 102, 111],
        'synonym_id::NCBI gene (formerly Entrezgene)': [9, 102, 111],
    },
}

print('The shipped selection:', DEFAULT_SS_EXTERNALS == DEFAULT_EXTERNALS[organism_ss])
make_configuration(organism_ss, DEFAULT_SS_EXTERNALS, template_ss, modified_ss)

The shipped selection: True


Enabled 64 (form, database, assembly) entries; wrote sus_scrofa_externals_modified.yml
Allowlisted (form, database) pairs the template does not have: [('transcript', 'EntrezGene'), ('transcript', 'NCBI gene (formerly Entrezgene)'), ('transcript', 'synonym_id::NCBI gene (formerly Entrezgene)')]
Same as the shipped configuration: True


In [13]:
_ = dm_ss.external_inst.load_modified_yaml()
print('Selected databases:', sorted(dm_ss.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_ss.external_inst.give_list_for_case('assembly')))

Selected databases: ['EntrezGene', 'HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'UniProtKB Gene Name', 'VGNC Symbol', 'miRBase', 'miRBase label', 'synonym_id::NCBI gene (formerly Entrezgene)']
Assemblies in play: [111]


## 6.5 — Rhesus and cynomolgus macaques

### 6.5.1 Rhesus macaque (*Macaca mulatta*)

Three assemblies in idtrack's window (release 48 on), one per release:
- `1` = MMUL_1 (releases 48–85)
- `801` = Mmul_8.0.1 (releases 86–97)
- `10` = Mmul_10 (releases 98 on)

> **Note:** Ensembl names both the MMUL_1 and the Mmul_10 core databases `macaca_mulatta_core_<release>_10`. idtrack gives MMUL_1 its own key `1` (its table entry says `DbSuffix: "10"` and `ReleaseRange: [48, 85]`), so the two genomes are never mixed; the configuration's assembly keys are therefore `1`, `801` and `10`.

The symbol source changes with the assembly: MMUL_1 and Mmul_8.0.1 genes carry **HGNC Symbol**, Mmul_10 genes **VGNC Symbol** (HGNC symbols are rare there). The selection holds both, and the identifier databases of the other organisms; names that no release of rhesus has (clone-based names, CCDS, Vega, Havana) are skipped.

In [14]:
organism_mmul = idtrack.organisms.resolve_organism('rhesus')  # 'Macaca mulatta' and 9544 work too
SNAPSHOT_RELEASE_MMUL = 115
dm_mmul = API.database_manager(organism_mmul, SNAPSHOT_RELEASE_MMUL, str(LOCAL_REPOSITORY))
template_mmul = DEFAULT_CONFIG / f'{organism_mmul}_externals_template.yml'
modified_mmul = Path(dm_mmul.external_inst.file_name_modified_yaml(location='configured'))

2026-11-24 11:53:04 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of macaca_mulatta assembly 10 (core suffix '10') for releases 98-116.


In [15]:
DEFAULT_MMUL_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [1, 801, 10],
        'Clone_based_vega_gene': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'HGNC Symbol': [1, 801, 10],
        'VGNC Symbol': [1, 801, 10],
        'Havana gene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'RFAM': [1, 801, 10],
        'RFAM label': [1, 801, 10],
        'miRBase': [1, 801, 10],
        'miRBase label': [1, 801, 10],
        'UniProtKB Gene Name': [1, 801, 10],
        'Vega gene': [1, 801, 10],
        'Vega_gene': [1, 801, 10],
        'synonym_id::HGNC Symbol': [1, 801, 10],
        'synonym_id::VGNC Symbol': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::UniProtKB Gene Name': [1, 801, 10],
    },
    'transcript': {
        'CCDS': [1, 801, 10],
        'CCDS label': [1, 801, 10],
        'Havana transcript': [1, 801, 10],
        'RefSeq_mRNA': [1, 801, 10],
        'RefSeq_mRNA_predicted': [1, 801, 10],
        'RefSeq_ncRNA': [1, 801, 10],
        'RefSeq_ncRNA_predicted': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
    },
    'translation': {
        'Havana translation': [1, 801, 10],
        'RefSeq_peptide': [1, 801, 10],
        'RefSeq_peptide_predicted': [1, 801, 10],
        'Uniprot/SPTREMBL': [1, 801, 10],
        'Uniprot/SWISSPROT': [1, 801, 10],
        'EntrezGene': [1, 801, 10],
        'NCBI gene (formerly Entrezgene)': [1, 801, 10],
        'synonym_id::NCBI gene (formerly Entrezgene)': [1, 801, 10],
    },
}

print('The shipped selection:', DEFAULT_MMUL_EXTERNALS == DEFAULT_EXTERNALS[organism_mmul])
make_configuration(organism_mmul, DEFAULT_MMUL_EXTERNALS, template_mmul, modified_mmul)

The shipped selection: True


Enabled 64 (form, database, assembly) entries; wrote macaca_mulatta_externals_modified.yml
Allowlisted (form, database) pairs the template does not have: [('gene', 'Clone_based_ensembl_gene'), ('gene', 'Clone_based_vega_gene'), ('gene', 'Havana gene'), ('gene', 'Vega gene'), ('gene', 'Vega_gene'), ('transcript', 'CCDS'), ('transcript', 'CCDS label'), ('transcript', 'Havana transcript'), ('transcript', 'synonym_id::NCBI gene (formerly Entrezgene)'), ('translation', 'Havana translation')]
Same as the shipped configuration: True


In [16]:
_ = dm_mmul.external_inst.load_modified_yaml()
print('Selected databases:', sorted(dm_mmul.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mmul.external_inst.give_list_for_case('assembly')))

Selected databases: ['EntrezGene', 'HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'UniProtKB Gene Name', 'VGNC Symbol', 'miRBase', 'miRBase label', 'synonym_id::NCBI gene (formerly Entrezgene)', 'synonym_id::UniProtKB Gene Name', 'synonym_id::VGNC Symbol']
Assemblies in play: [10]


### 6.5.2 Cynomolgus macaque (*Macaca fascicularis*)

Two assemblies, one per release, the change at release 103:
- `5` = Macaca_fascicularis_5.0 (releases 91–102)
- `6` = Macaca_fascicularis_6.0 (releases 103 on)

As for rhesus, 5.0 genes carry **HGNC Symbol** and 6.0 genes **VGNC Symbol**, but Ensembl's symbols of 6.0 are few (a few hundred genes of each kind); NCBI's gene ids and UniProt's gene names cover most genes. Ensembl lists cynomolgus under the common name "Crab-eating macaque".

In [17]:
organism_mfas = idtrack.organisms.resolve_organism('cynomolgus')  # 'Crab-eating macaque' and 9541 work with API.ensembl_organism
SNAPSHOT_RELEASE_MFAS = 115
dm_mfas = API.database_manager(organism_mfas, SNAPSHOT_RELEASE_MFAS, str(LOCAL_REPOSITORY))
template_mfas = DEFAULT_CONFIG / f'{organism_mfas}_externals_template.yml'
modified_mfas = Path(dm_mfas.external_inst.file_name_modified_yaml(location='configured'))

2026-11-24 11:53:07 INFO:idtrack.build.database_manager: Scanning the HTTPS dumps of macaca_fascicularis assembly 6 (core suffix '6') for releases 103-116.


In [18]:
DEFAULT_MFAS_EXTERNALS = {
    'gene': {
        'Clone_based_ensembl_gene': [5, 6],
        'Clone_based_vega_gene': [5, 6],
        'EntrezGene': [5, 6],
        'HGNC Symbol': [5, 6],
        'VGNC Symbol': [5, 6],
        'Havana gene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'RFAM': [5, 6],
        'RFAM label': [5, 6],
        'miRBase': [5, 6],
        'miRBase label': [5, 6],
        'UniProtKB Gene Name': [5, 6],
        'Vega gene': [5, 6],
        'Vega_gene': [5, 6],
        'synonym_id::HGNC Symbol': [5, 6],
        'synonym_id::VGNC Symbol': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::UniProtKB Gene Name': [5, 6],
    },
    'transcript': {
        'CCDS': [5, 6],
        'CCDS label': [5, 6],
        'Havana transcript': [5, 6],
        'RefSeq_mRNA': [5, 6],
        'RefSeq_mRNA_predicted': [5, 6],
        'RefSeq_ncRNA': [5, 6],
        'RefSeq_ncRNA_predicted': [5, 6],
        'EntrezGene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
    },
    'translation': {
        'Havana translation': [5, 6],
        'RefSeq_peptide': [5, 6],
        'RefSeq_peptide_predicted': [5, 6],
        'Uniprot/SPTREMBL': [5, 6],
        'Uniprot/SWISSPROT': [5, 6],
        'EntrezGene': [5, 6],
        'NCBI gene (formerly Entrezgene)': [5, 6],
        'synonym_id::NCBI gene (formerly Entrezgene)': [5, 6],
    },
}

print('The shipped selection:', DEFAULT_MFAS_EXTERNALS == DEFAULT_EXTERNALS[organism_mfas])
make_configuration(organism_mfas, DEFAULT_MFAS_EXTERNALS, template_mfas, modified_mfas)

The shipped selection: True
Enabled 35 (form, database, assembly) entries; wrote macaca_fascicularis_externals_modified.yml
Allowlisted (form, database) pairs the template does not have: [('gene', 'Clone_based_ensembl_gene'), ('gene', 'Clone_based_vega_gene'), ('gene', 'Havana gene'), ('gene', 'miRBase'), ('gene', 'miRBase label'), ('gene', 'Vega gene'), ('gene', 'Vega_gene'), ('transcript', 'CCDS'), ('transcript', 'CCDS label'), ('transcript', 'Havana transcript'), ('transcript', 'RefSeq_ncRNA'), ('transcript', 'EntrezGene'), ('transcript', 'NCBI gene (formerly Entrezgene)'), ('transcript', 'synonym_id::NCBI gene (formerly Entrezgene)'), ('translation', 'Havana translation'), ('translation', 'EntrezGene'), ('translation', 'NCBI gene (formerly Entrezgene)'), ('translation', 'synonym_id::NCBI gene (formerly Entrezgene)')]
Same as the shipped configuration: True


In [19]:
_ = dm_mfas.external_inst.load_modified_yaml()
print('Selected databases:', sorted(dm_mfas.external_inst.give_list_for_case('db')))
print('Assemblies in play:', sorted(dm_mfas.external_inst.give_list_for_case('assembly')))

Selected databases: ['EntrezGene', 'HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label', 'UniProtKB Gene Name', 'VGNC Symbol', 'synonym_id::NCBI gene (formerly Entrezgene)', 'synonym_id::UniProtKB Gene Name', 'synonym_id::VGNC Symbol']
Assemblies in play: [6]


## 6.6 — Extending a configuration to a new Ensembl release

Each Ensembl release can add or drop external databases, and a configuration lists, per database and assembly, the releases it was made from. A graph whose window includes a release that the configuration does not list gets **no external identifiers for that release**; idtrack warns about it once when it builds the graph.

You do not need to make the whole file again. `ExternalDatabases.extend_modified_yaml(content)` takes the database-content table of the new release(s), keeps a copy of the file it replaces (`<file>.bak`), and:
- adds the new release(s) to every entry the table holds, keeping every `Include` flag as you set it;
- adds databases the file does not know yet with `Include: false` and lists them, so that you can decide whether to select them;
- adds a known database that appears with a new assembly, with the database's flag when its other assemblies agree.

The content table comes from a manager whose window is just the new release(s), which keeps the download to one release per assembly. The extended file is written to your local repository (or to `output_path=`). The cell below does it for Ensembl's newest release when `EXTEND = True` (a download of that release's tables: minutes for human); the shipped files already cover releases up to 116.

In [20]:
%%collapse Click to show the log
EXTEND = False  # True: download the newest release's tables and extend the human configuration of 6.2

if EXTEND:
    _, NEW_RELEASE = API.ensembl_organism('human')  # Ensembl's newest release
    dm_new = idtrack.build.DatabaseManager(
        organism='homo_sapiens',
        form='gene',
        local_repository=str(LOCAL_REPOSITORY),
        first_release=NEW_RELEASE,
        snapshot_release=NEW_RELEASE,
    )
    content_new = dm_new.create_database_content()
    report = dm_new.external_inst.extend_modified_yaml(content_new)
    print('Written:', report['path'])
    print('Releases added:', report['releases'], '| entries extended:', report['extended_entries'])
    print('New databases (Include: false, review them):', report['new_databases'])
    print('New assembly entries of known databases:', report['new_assembly_entries'])
else:
    print('Not run (EXTEND = False).')

## 6.7 — Adding an organism

idtrack supports human, mouse, pig, rhesus macaque and cynomolgus macaque. Another Ensembl species needs **two things**:

1. **An entry in the organism table** (`ORGANISM_TABLE` in `idtrack/organisms.py`): one entry per assembly, keyed by an integer, with the MySQL `Ports`, a `Priority` (1 = preferred), the `ReleaseRange` of the assembly and, optionally, its Ensembl `Name` and `DbSuffix` (the code in the core database names `<organism>_core_<release>_<code>`, when it differs from the key). To try an organism without editing the package, register it for the running session with `idtrack.organisms.register_organism(name, assemblies, symbol_databases, aliases)`; it checks the entry and updates everything derived from the table. Direct MySQL access is optional: where the ports are blocked, idtrack reads the same tables from the HTTPS dumps.
2. **An externals configuration**: make a template and apply an allowlist, as for the organisms above.

idtrack reads Ensembl ids with their versions: an organism whose Ensembl ids carry no versions is not supported (a graph build refuses it with an error that says so).

A recipe:

- Step A: resolve the Ensembl species name with `idtrack.API.ensembl_organism(...)`.
- Step B: find its assemblies in the dump listings (`https://ftp.ensembl.org/pub/release-<n>/mysql/`) and register them (example below). Listing every assembly is what lets identifiers of older builds convert, and databases be read per assembly.
- Step C: make the template (`DatabaseManager.create_database_content()` → `create_template_yaml(...)`) and apply an allowlist.
- Step D: build a graph (Part 2) and run its self-tests (Appendix).

The full recipe (with the tests and documentation to update when an organism is added to idtrack itself): [Organisms](../organisms.rst).

> **Warning:** Until Step B is done, `DatabaseManager` raises `NotImplementedError` for that organism.

In [21]:
# Is an organism supported by this version of idtrack? (Changes nothing.)
from idtrack import organisms

print('Supported organisms:', ', '.join(organisms.supported_organisms()))

# Resolve a name with Ensembl REST (works for any Ensembl species)
formal_name, latest_release = API.ensembl_organism('danio rerio')  # zebrafish
print('danio rerio ->', formal_name, '| Ensembl newest release:', latest_release)
print('Supported here:', formal_name in organisms.supported_organisms())

2026-11-24 11:53:09 INFO:idtrack.build.species: Ensembl Rest API query to get the organism names and associated releases.


Supported organisms: homo_sapiens, macaca_fascicularis, macaca_mulatta, mus_musculus, sus_scrofa


danio rerio -> danio_rerio | Ensembl newest release: 116
Supported here: False


In [22]:
# Register zebrafish (GRCz11 core databases are named danio_rerio_core_<release>_11) for this Python session.
# Check the release range against the dump listings before relying on it.
organisms.register_organism(
    'danio_rerio',
    {11: {'Name': 'GRCz11', 'Ports': [3306, 5306], 'Priority': 1, 'ReleaseRange': [92, None]}},
    symbol_databases=['ZFIN'],
)
print('danio_rerio' in organisms.supported_organisms(), sorted(organisms.ORGANISM_TABLE['danio_rerio']))

True [11]


## 6.8 — Checklist and good practice

Your local repository now holds these files (or the shipped ones serve the organisms you skipped). A graph built from them in this local repository (Part 2) records the selection it was built from.

In [23]:
for organism in ['homo_sapiens', 'mus_musculus', 'sus_scrofa', 'macaca_mulatta', 'macaca_fascicularis']:
    path = LOCAL_REPOSITORY / f'{organism}_externals_modified.yml'
    print('OK     ' if path.exists() else 'MISSING', path.name)

OK      homo_sapiens_externals_modified.yml
OK      mus_musculus_externals_modified.yml
OK      sus_scrofa_externals_modified.yml
OK      macaca_mulatta_externals_modified.yml
OK      macaca_fascicularis_externals_modified.yml


1. **Keep selections small**: many overlapping databases make answers more ambiguous.
2. **Several selections** (a "strict" and a "broad" one, say): one local repository per selection, since the file name is fixed per organism and a graph file records the selection it was built from (loading it for another selection is an error).
3. **Assemblies**:
   - in Ensembl's database names, the last number is the genome assembly (human 38 = GRCh38, mouse 39 = GRCm39, pig 111 = Sscrofa11.1); idtrack's assembly keys are these codes, except rhesus MMUL_1 (key `1`), whose databases share the code `10` with Mmul_10;
   - assembly keys are species-specific, and a database can be listed for some assemblies only;
   - keeping every assembly block selected is usually right when you integrate datasets of mixed builds.
4. **New Ensembl releases**: extend your configuration with `extend_modified_yaml` (6.6) instead of making it again, and review the databases it reports as new.